# Phase 2 — Figures & Plots (reads saved Drive logs)

Generates all publication figures from the artifacts already saved by the experiment notebooks. **Does not re-run any training** — it only reads JSON/npy logs. Safe to run anytime, re-runnable for updated figures.

Produces, saved to `Phase2_Collapse_Study/figures/`:
1. SSL pretraining curves (VICReg & MAE: loss, emb_std, invariance)
2. Main comparison — gate weight / similarity std / accuracy / MRI-zeroed drop across all conditions
3. Gate-weight composition (MRI/clin/bio) per condition
4. Per-condition training curves (val MCC)
5. Summary results table (CSV + PNG)

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import os, json, glob, numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({'figure.dpi':120,'font.size':11,'axes.grid':True,'grid.alpha':0.3})

DRIVE='/content/drive/My Drive/ADNI_NewDS/'
PHASE2=os.path.join(DRIVE,'Phase2_Collapse_Study')
SSL_DIR=os.path.join(PHASE2,'ssl_pretrain')
RESULTS_DIR=os.path.join(PHASE2,'results')
LAB_LOG=os.path.join(PHASE2,'phase2_lab_log.jsonl')
FIG_DIR=os.path.join(PHASE2,'figures'); os.makedirs(FIG_DIR, exist_ok=True)
print('Figures will be saved to:', FIG_DIR)


## 1. SSL pretraining curves (VICReg & MAE)

In [ ]:
def load_json(p):
    try: return json.load(open(p))
    except: return None
vic=load_json(os.path.join(SSL_DIR,'vicreg_log.json'))
mae=load_json(os.path.join(SSL_DIR,'mae_log.json'))

if vic:
    ep=[r['epoch'] for r in vic]
    fig,ax=plt.subplots(1,3,figsize=(15,4))
    ax[0].plot(ep,[r['loss'] for r in vic],'-o',ms=3,color='#2E75B6'); ax[0].set_title('VICReg total loss'); ax[0].set_xlabel('epoch')
    ax[1].plot(ep,[r['emb_std'] for r in vic],'-o',ms=3,color='#2E8B57'); ax[1].axhline(0.01,ls='--',c='r',label='collapse threshold'); ax[1].set_title('Embedding STD (collapse monitor)'); ax[1].set_xlabel('epoch'); ax[1].legend()
    if 'inv' in vic[0]: ax[2].plot(ep,[r['inv'] for r in vic],'-o',ms=3,color='#E8A33D'); ax[2].set_title('Invariance loss (view agreement)'); ax[2].set_xlabel('epoch')
    plt.suptitle('VICReg SSL Pretraining',fontweight='bold'); plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR,'fig1a_vicreg_pretraining.png'),bbox_inches='tight'); plt.show()
else: print('No vicreg_log.json yet (pretraining still running or not done).')

if mae:
    ep=[r['epoch'] for r in mae]
    fig,ax=plt.subplots(1,2,figsize=(10,4))
    ax[0].plot(ep,[r['loss'] for r in mae],'-o',ms=3,color='#2E75B6'); ax[0].set_title('MAE reconstruction loss'); ax[0].set_xlabel('epoch')
    ax[1].plot(ep,[r['emb_std'] for r in mae],'-o',ms=3,color='#2E8B57'); ax[1].axhline(0.01,ls='--',c='r'); ax[1].set_title('Embedding STD'); ax[1].set_xlabel('epoch')
    plt.suptitle('Fixed-MAE SSL Pretraining',fontweight='bold'); plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR,'fig1b_mae_pretraining.png'),bbox_inches='tight'); plt.show()
else: print('No mae_log.json yet.')


## 2. Main comparison across all conditions

In [ ]:
# read lab log
entries=[]
if os.path.exists(LAB_LOG):
    for line in open(LAB_LOG):
        line=line.strip()
        if line: entries.append(json.loads(line))
# add Condition 1 baseline (from Phase 1 audit, not in log)
baseline={'condition':'cond1_baseline','diagnostic':{'gate':{'mri':0.0,'clin':1.0,'bio':0.0},
  'ablation':{'full':{'acc':0.8966},'mri':{'acc':0.8966}},'img_sim_std':0.0}}
all_c=[baseline]+entries

def short(name):
    return (name.replace('condition','C').replace('_monai_pretrained','').replace('_brain_preprocessed','')
            .replace('_brain_finetuned','-FT').replace('cond1_baseline','C1-base')
            .replace('_ssl_','-SSL-').replace('_frozen','').replace('_UNFROZEN','-unfroz')[:16])
labels=[short(c['condition']) for c in all_c]
gate=[c['diagnostic']['gate']['mri'] for c in all_c]
simstd=[c['diagnostic']['img_sim_std'] for c in all_c]
acc=[c['diagnostic']['ablation']['full']['acc'] for c in all_c]
drop=[c['diagnostic']['ablation']['full']['acc']-c['diagnostic']['ablation']['mri']['acc'] for c in all_c]
cols=['#999999']+['#2E8B57' if g>=0.10 and d>0.01 else '#D9544D' for g,d in zip(gate[1:],drop[1:])]

fig,ax=plt.subplots(2,2,figsize=(14,9))
for a,(vals,title,thr) in zip(ax.flat,[(gate,'MRI Gate Weight',0.10),(simstd,'Image Similarity STD (no-collapse)',0.01),(acc,'Full Accuracy',None),(drop,'Accuracy Drop when MRI Zeroed',0.01)]):
    a.bar(range(len(vals)),vals,color=cols); a.set_title(title,fontweight='bold')
    a.set_xticks(range(len(labels))); a.set_xticklabels(labels,rotation=45,ha='right',fontsize=8)
    if thr: a.axhline(thr,ls='--',c='gray',lw=1)
    for i,v in enumerate(vals): a.text(i,v,f'{v:.3f}',ha='center',va='bottom',fontsize=7)
plt.suptitle('Phase 2: Imaging-Branch Recovery Across Conditions',fontsize=13,fontweight='bold'); plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR,'fig2_main_comparison.png'),bbox_inches='tight'); plt.show()


## 3. Gate-weight composition per condition

In [ ]:
mri=[c['diagnostic']['gate']['mri'] for c in all_c]
clin=[c['diagnostic']['gate']['clin'] for c in all_c]
bio=[c['diagnostic']['gate']['bio'] for c in all_c]
fig,ax=plt.subplots(figsize=(11,5))
x=range(len(labels))
ax.bar(x,mri,label='MRI',color='#2E8B57')
ax.bar(x,clin,bottom=mri,label='Clinical',color='#2E75B6')
ax.bar(x,bio,bottom=[m+c for m,c in zip(mri,clin)],label='Biomarker',color='#E8A33D')
ax.set_xticks(list(x)); ax.set_xticklabels(labels,rotation=45,ha='right',fontsize=8)
ax.set_ylabel('gate weight'); ax.set_title('Fusion Gate Composition by Modality',fontweight='bold'); ax.legend()
plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR,'fig3_gate_composition.png'),bbox_inches='tight'); plt.show()


## 4. Per-condition training curves (val MCC)

In [ ]:
hist_files=glob.glob(os.path.join(RESULTS_DIR,'*','training_history.json'))
if hist_files:
    fig,ax=plt.subplots(figsize=(10,5))
    for hf in sorted(hist_files):
        cond=os.path.basename(os.path.dirname(hf))
        h=load_json(hf)
        if h: ax.plot([r['epoch'] for r in h],[r['val_mcc'] for r in h],'-o',ms=3,label=short(cond))
    ax.set_xlabel('epoch'); ax.set_ylabel('validation MCC'); ax.set_title('Training Curves by Condition',fontweight='bold'); ax.legend(fontsize=8)
    plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR,'fig4_training_curves.png'),bbox_inches='tight'); plt.show()
else: print('No training_history.json files found yet.')


## 5. Summary table (CSV + PNG)

In [ ]:
import csv
rows=[]
for c in all_c:
    d=c['diagnostic']
    rows.append({'condition':c['condition'],'mri_gate':round(d['gate']['mri'],4),
        'clin_gate':round(d['gate']['clin'],4),'bio_gate':round(d['gate']['bio'],4),
        'sim_std':round(d['img_sim_std'],4),'full_acc':round(d['ablation']['full']['acc'],4),
        'mri_zeroed_acc':round(d['ablation']['mri']['acc'],4),
        'mri_contribution':round(d['ablation']['full']['acc']-d['ablation']['mri']['acc'],4)})
# CSV
with open(os.path.join(FIG_DIR,'summary_table.csv'),'w',newline='') as f:
    w=csv.DictWriter(f,fieldnames=rows[0].keys()); w.writeheader(); w.writerows(rows)
# PNG table
fig,ax=plt.subplots(figsize=(13,0.6*len(rows)+1)); ax.axis('off')
cols=list(rows[0].keys())
tbl=ax.table(cellText=[[r[c] for c in cols] for r in rows],colLabels=cols,loc='center',cellLoc='center')
tbl.auto_set_font_size(False); tbl.set_fontsize(8); tbl.scale(1,1.5)
for j in range(len(cols)): tbl[0,j].set_facecolor('#2E75B6'); tbl[0,j].set_text_props(color='white',weight='bold')
plt.title('Phase 2 Summary — All Conditions',fontweight='bold',pad=20)
plt.savefig(os.path.join(FIG_DIR,'fig5_summary_table.png'),bbox_inches='tight',dpi=120); plt.show()
print('\nAll figures saved to', FIG_DIR)
for r in rows: print(r)
